# 03 · YOLO Concept —— 网格一眼看全 + NMS 消重的艺术

**家族位置**：`03_CNN_Segmentation_Detection` 第 3 站（检测·思想篇）。01 自训了分割（涂色，mIoU）、02 用预训练验了检测（画框，mAP）；本章不训大模型，只把**最快的检测思想 YOLO**掰开揉碎——为什么叫 You Only Look Once，一眼怎么够。

**学习目标**
1. YOLO 的 grid 思想：把图切成 SxS 格子，每格直接猜框+类，一步到位为什么快
2. NMS 阈值的艺术：同堆框，IoU 0.3/0.5/0.9 消重结果天差地别
3. 分数阈值 vs IoU 阈值：两个旋钮如何联动决定"检出多少"与"检得准不准"
4. Anchor 有无的取舍：预设框 vs 直接回归，YOLO v1 的简洁与代价

## 1. 原理：YOLO 为什么能实时

### 通俗理解

**一句话**：Faster 是"先圈一堆候选(2000个)再一个个精审"，YOLO 是"把图切成 7×7=49 个格子，每个格子当场拍板——我这有没有物体、框在哪"。少了候选这一趟，自然快。

**比喻**：Faster 像老师先让全班举手再逐个提问；YOLO 像把教室切成 49 个座位区，每区派个代表直接喊"我这有只猫，框在这"。49 次喊话 vs 2000 次筛选，速度差一个量级。

### 结构账

```
Faster： 图像 → backbone→FPN → RPN(Anchor 粗筛2000框) → RoI Head 精修 → NMS    (两步)
Retina：  图像 → backbone→FPN → 每层Anchor 直接回归+分类(Focal) → NMS              (一步，但仍多尺度Anchor)
YOLO v1： 图像 → backbone(单尺度) → 7×7 Grid，每格 B=2 框 + C 类 + 置信度 → NMS      (一步，极简)
```

- **Grid**：SxS 网格，物体中心落哪格就归哪格管
- **置信度**：`conf = Pr(有物体) × IoU(预测框,真框)`，既管"有没有"又管"框准不准"
- **NMS 仍要**：49格×2框=98框，重叠仍需消重，IoU 阈值是第二道闸
- **代价**：单尺度网格，小目标/密集目标吃亏——这才有了后来的多尺度 YOLO

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from PIL import Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import _resolve_pet_root, _collect_pet_samples
from common.utils import set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

def box_iou(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = (a[2]-a[0])*(a[3]-a[1]); area_b = (b[2]-b[0])*(b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

def nms_numpy(boxes, scores, iou_th=0.5):
    order = np.argsort(-scores)
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(int(i))
        rest = []
        for j in order[1:]:
            if box_iou(boxes[i], boxes[j]) <= iou_th:
                rest.append(j)
        order = np.array(rest, dtype=int)
    return keep


## 2. 数据：1 张 Pet 图作网格画布 + 合成框演示 NMS

重用 01/02 已下载的 Pet 图（`Abyssinian_100`），不做训练——一张图即讲清 grid 归属；NMS 用合成拥挤框，可控可复现。

In [ ]:
pet_root = _resolve_pet_root(str(ROOT / "data"))
names = _collect_pet_samples(pet_root)
# 固定 Abyssinian_100，与 02 站一致
target = "Abyssinian_100" if "Abyssinian_100" in names else names[0]
print("画布:", target)
pil = Image.open(pet_root / "images" / f"{target}.jpg").convert("RGB")
W, H = pil.size
print(f"原图尺寸 {W}×{H}")

# fig0：grid 思想——7×7 网格叠加 + 模拟真框与归属格子
S = 7
# 模拟 2 个真框（归一化→像素）：一猫主体 + 一小物体（呼应密集/小目标痛点）
gt_boxes = np.array([[0.30, 0.35, 0.72, 0.85],[0.68, 0.18, 0.92, 0.42]])  # x1,y1,x2,y2 归一化
# 物体中心落在第几格
def center_grid(box, S):
    cx = (box[0]+box[2])/2; cy = (box[1]+box[3])/2
    gx = int(cx * S); gy = int(cy * S)
    return gx, gy, cx, cy

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.imshow(pil)
ax.set_title(f"YOLO Grid 思想：{S}×{S} 网格，中心落哪格归哪格管", fontsize=10)
# 画网格
for i in range(1, S):
    ax.axvline(i*W/S, color="white", linewidth=0.8, alpha=0.85)
    ax.axhline(i*H/S, color="white", linewidth=0.8, alpha=0.85)
cmap = plt.cm.get_cmap("tab10")
for idx, b in enumerate(gt_boxes):
    x1,y1,x2,y2 = b * np.array([W, H, W, H])
    gx, gy, cx, cy = center_grid(b, S)
    # 负责格子高亮
    ax.add_patch(patches.Rectangle((gx*W/S, gy*H/S), W/S, H/S, linewidth=2.2, edgecolor=cmap(idx), facecolor=cmap(idx), alpha=0.18))
    # 真框
    ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2.5, edgecolor=cmap(idx), facecolor="none"))
    ax.plot(cx*W, cy*H, marker="*", color=cmap(idx), markersize=14, markeredgecolor="white", markeredgewidth=0.8)
    ax.text(x1, max(0,y1-6), f"GT{idx+1} 归格({gx},{gy})", fontsize=8, color="white", bbox=dict(facecolor=cmap(idx), alpha=0.9, pad=2, edgecolor="none"))
ax.axis("off")
plt.tight_layout()
plt.savefig(FIGS / "fig0_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("grid 归属:", [center_grid(b,S)[:2] for b in gt_boxes])


## 3. 主实验 A：NMS 阈值 sweep——同堆框，三种命运

In [ ]:
# 合成拥挤框：两簇重叠（模拟同一只猫被多格/多Anchor 猜中）+ 2 孤立
boxes = np.array([[30,30,110,110],[38,38,118,118],[36,32,112,108],[44,40,120,116],
                  [150,150,230,230],[158,158,238,238],[250,40,310,100],[252,42,312,102]], dtype=float)
scores = np.array([0.96, 0.92, 0.88, 0.85, 0.94, 0.89, 0.91, 0.87])
print("合成 8 框，分数", [round(float(s),2) for s in scores])

ths = [0.3, 0.5, 0.9]
keeps = {t: nms_numpy(boxes, scores, t) for t in ths}
for t in ths:
    print(f"IoU>{t}: keep {keeps[t]}  保留 {len(keeps[t])}  分数 {[round(float(scores[i]),2) for i in keeps[t]]}")

# fig1：1×3，阈值越严越狠，越松越放
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharex=True, sharey=True)
for ax, t in zip(axes, ths):
    ax.set_xlim(0, 340); ax.set_ylim(260, 0)
    ax.set_aspect("equal")
    ax.set_title(f"NMS IoU>{t}  保留 {len(keeps[t])}/8", fontsize=10)
    # 全框淡色底
    for k in range(len(boxes)):
        x1,y1,x2,y2 = boxes[k]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=1, edgecolor="#BBBBBB", facecolor="#BBBBBB", alpha=0.14))
    # 保留框高亮
    cmap = plt.cm.get_cmap("tab10")
    for rank, k in enumerate(keeps[t]):
        x1,y1,x2,y2 = boxes[k]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2.4, edgecolor=cmap(rank%10), facecolor="none"))
        ax.text(x1, y1-4, f"{scores[k]:.2f}", fontsize=7, color=cmap(rank%10), weight="bold")
plt.suptitle("同堆框，NMS 阈值决定命运：0.3严(去重狠) / 0.5中 / 0.9松(几乎不去)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_nms_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：阈值-保留数曲线
xs = np.linspace(0.05, 0.95, 19)
ys = [len(nms_numpy(boxes, scores, float(x))) for x in xs]
fig, ax = plt.subplots(figsize=(6.2, 4))
ax.plot(xs, ys, marker="o", ms=4, color="#4C72B0")
for t in ths:
    ax.axvline(t, color="#DD8452", linestyle="--", linewidth=1)
    ax.text(t, len(keeps[t])+0.15, f"{t}→{len(keeps[t])}", ha="center", fontsize=8, color="#DD8452")
ax.set_xlabel("NMS IoU 阈值"); ax.set_ylabel("保留框数")
ax.set_title("NMS 阈值 vs 保留数（阈值↑越宽松，保留越多）")
ax.set_ylim(0, 9)
plt.tight_layout()
plt.savefig(FIGS / "fig2_nms_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 主实验 B：分数阈值 vs NMS——两个旋钮的联动

NMS 管"重叠去重"，分数阈值管"多弱算检出"。同堆框扫分数阈值，看 precision/recall/F1 权衡（以 8 框中设 3 个真物体为 GT，演示用）。

In [ ]:
# 设 GT 3 个（两簇各一 + 孤立一），用 IoU>0.5 判 TP
GT = np.array([[32,32,112,112],[152,152,232,232],[251,41,311,101]], dtype=float)
# 预测 8 框同上，分数同上；NMS 固定 0.5 后再扫分数阈值
keep05 = nms_numpy(boxes, scores, 0.5)
# 按分数阈值扫
ths_sc = np.linspace(0.80, 0.97, 18)
prec, rec, f1 = [], [], []
for th in ths_sc:
    # NMS 后再过分数阈值
    cur_idx = [k for k in keep05 if scores[k] >= th]
    # 贪心匹配 GT
    matched = set()
    tp = 0
    for k in sorted(cur_idx, key=lambda i: -scores[i]):
        best, bj = 0, -1
        for j, g in enumerate(GT):
            if j in matched: continue
            iou = box_iou(boxes[k], g)
            if iou > best: best, bj = iou, j
        if best >= 0.5:
            tp += 1; matched.add(bj)
    fp = len(cur_idx) - tp; fn = len(GT) - tp
    p = tp / (tp+fp) if tp+fp else 1.0
    r = tp / (tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    prec.append(p); rec.append(r); f1.append(f)

# 打印三档
for th in [0.85, 0.90, 0.93]:
    idx = int(np.argmin(np.abs(ths_sc - th)))
    print(f"score≥{th:.2f}: P={prec[idx]:.3f} R={rec[idx]:.3f} F1={f1[idx]:.3f}")

fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(ths_sc, prec, label="Precision", color="#4C72B0", marker="o", ms=3)
ax.plot(ths_sc, rec, label="Recall", color="#55A868", marker="o", ms=3)
ax.plot(ths_sc, f1, label="F1", color="#C44E52", linewidth=2, marker="o", ms=3)
ax.set_xlabel("分数阈值"); ax.set_ylabel("分数")
ax.set_title("分数阈值 vs P/R/F1（NMS 0.5 固定，GT=3）")
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(FIGS / "fig3_pr.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. YOLO 解码演示：格子如何"猜"框

每格预测 `B` 个框的 `(tx,ty,tw,th,conf)` + `C` 类概率。演示：格子中心 + 偏移 → 真实框中心，宽高相对格子尺寸的指数解码（YOLO v1 式）。

In [ ]:
S, B = 7, 2
# 选上图 GT1 的负责格子 (2,2) 为例，演示该格 B=2 的两候选如何经 NMS 留一
gx, gy = 2, 2  # 来自 fig0 的 GT1 归属
# 模拟该格网络输出（已 sigmoid/exp 后）：相对格子的偏移与宽高
# 候选A：偏移小、贴合好、conf 高；候选B：偏移大、conf 低
preds = [
    {"tx": 0.62, "ty": 0.48, "tw": 1.42, "th": 1.35, "conf": 0.92, "cls": "cat"},
    {"tx": 0.15, "ty": 0.71, "tw": 1.10, "th": 1.05, "conf": 0.61, "cls": "cat"},
]
# 解码：cx = (gx+tx)/S, cy = (gy+ty)/S, w = (exp(tw)/S) 归一化宽度 的简化示意（此处 tw 已为相对格子的对数宽）
def decode(p, gx, gy, S):
    cx = (gx + p["tx"]) / S
    cy = (gy + p["ty"]) / S
    # 为演示直观，tw/th 直接当归一化宽高的小数（不做 exp，示意）
    w = p["tw"] / S * 0.55
    h = p["th"] / S * 0.62
    x1, y1, x2, y2 = (cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H
    return np.array([x1,y1,x2,y2]), (cx*W, cy*H)

fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.imshow(pil); ax.axis("off")
# 网格 + 负责格高亮
for i in range(1, S):
    ax.axvline(i*W/S, color="white", linewidth=0.7, alpha=0.8)
    ax.axhline(i*H/S, color="white", linewidth=0.7, alpha=0.8)
ax.add_patch(patches.Rectangle((gx*W/S, gy*H/S), W/S, H/S, linewidth=2.6, edgecolor="#FFD400", facecolor="#FFD400", alpha=0.22))
ax.text(gx*W/S+2, gy*H/S+10, f"负责格({gx},{gy})", fontsize=8, color="#7A6400", weight="bold", bbox=dict(facecolor="#FFD400", alpha=0.92, pad=2, edgecolor="none"))
# 两候选框 + 中心连线
cmap = plt.cm.get_cmap("tab10")
for i, p in enumerate(preds):
    box, (cx, cy) = decode(p, gx, gy, S)
    x1,y1,x2,y2 = box
    ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2.4 if i==0 else 1.8, edgecolor=cmap(i), facecolor="none", linestyle="-" if i==0 else "--"))
    ax.plot(cx, cy, marker="o", color=cmap(i), markersize=7, markeredgecolor="white", markeredgewidth=0.9)
    # 格子中心到预测中心的虚线
    gcx, gcy = (gx+0.5)*W/S, (gy+0.5)*H/S
    ax.plot([gcx, cx], [gcy, cy], color=cmap(i), linewidth=1.2, linestyle=":", alpha=0.9)
    ax.text(x1, y1-5, f"B{i+1} conf={p['conf']:.2f} tx={p['tx']:.2f} ty={p['ty']:.2f}", fontsize=7, color="white", bbox=dict(facecolor=cmap(i), alpha=0.92, pad=1.5, edgecolor="none"))
# 格子中心
ax.plot((gx+0.5)*W/S, (gy+0.5)*H/S, marker="+", color="#FFD400", markersize=14, markeredgewidth=2)
ax.set_title("YOLO 解码：格子中心 + (tx,ty) 偏移 → 框中心", fontsize=10)
plt.tight_layout()
plt.savefig(FIGS / "fig4_decode.png", dpi=150, bbox_inches="tight")
plt.show()
# NMS 消重演示（同格 B=2 必重叠）
b01 = decode(preds[0], gx, gy, S)[0]; b02 = decode(preds[1], gx, gy, S)[0]
print(f"同格 B=2 IoU={box_iou(b01,b02):.3f}  NMS 0.5 后保留 conf 高者 B1")


## 6. 总结与下一步

**本项目收获**

1. YOLO grid：一图 S×S 格子、每格 B 框，一眼看完全图，快在"无 RPN、单尺度、一次前向"
2. NMS 双阈值：IoU 阈值管"重叠算一个"，分数阈值管"多弱算检出"，两者联动定 P/R
3. 解码：格子中心 + tx/ty 偏移 + 宽高回归 → 绝对框，conf 同时管有无与准度
4. 取舍：单尺度快但小/密目标吃亏——后续 YOLO 用 FPN/多尺度、Anchor→Anchor-free 演进

**家族闭环**：分割(涂色)→检测(画框)→YOLO(快检)，02 的卷积骨干 + U-Net 的多尺度思想在此汇合；下一步可在全量 Pet 上以 YOLO 思路做小目标增强实验。

**下一步**：回头把本家族通俗讲解精简进 `../README.md`（家族级导读），并可把 02 家族 README 同步加"通俗理解"段落。